# Kodak-24: common benchmark for every reconstruction model

This is the team's single local evaluation entry point. It uses all 24 native-resolution Kodak images and produces one CSV row per method × operating point × image. Built-in RAW RGB24 and JPEG baselines use exactly the same timing and quality code as learned models.

**Metric rules:** PSNR/SSIM are RGB `[0,1]`; LPIPS is AlexNet; CBR is complex channel uses divided by `3×H×W`; compression ratio is raw RGB24 bits divided by serialized payload bits. JPEG/RAW do not claim a CBR unless a digital PHY is explicitly supplied. No evaluation-side resizing is allowed.

## 1. Environment

Create the repository's Python 3.12 environment and install `requirements.txt` before opening this notebook. If this kernel is new, run `%pip install -e ..[benchmark,metrics]` once and restart it. LPIPS may download its pretrained AlexNet weights on first use.

In [1]:
from pathlib import Path
import getpass, math, os, random, subprocess, sys

import numpy as np
import pandas as pd
from PIL import Image
import torch

candidate = Path.cwd().resolve()
PROJECT_ROOT = candidate if (candidate / 'pyproject.toml').is_file() else candidate.parent
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Start Jupyter in the repository root or its Notebooks directory.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from Evaluation.kodak_benchmark import (
    CallableCodec, EncDecCodec, JPEGCodec, RawRGBCodec, append_rows_csv,
    benchmark_codec, dataset_files, environment_summary, make_run_id,
    summarize_rows,
)
environment_summary()

{'python': '3.13.4',
 'platform': 'Windows-11-10.0.26200-SP0',
 'torch': '2.14.0+cpu',
 'cuda': 'None',
 'device': 'CPU'}

In [2]:
# Team/run configuration. Keep one RUN_ID for one complete execution.
CONTRIBUTOR = os.getenv('KODAK_BENCHMARK_CONTRIBUTOR', getpass.getuser())
RUN_ID = os.getenv('KODAK_BENCHMARK_RUN_ID', make_run_id())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
KODAK_DIR = PROJECT_ROOT / 'data' / 'Kodak24'
CSV_PATH = PROJECT_ROOT / 'results' / 'kodak24_benchmark.csv'
JPEG_QUALITIES = [int(value) for value in os.getenv('KODAK_JPEG_QUALITIES', '20,30,40,50,60,70,80,90').split(',')]
TIMING_REPETITIONS = int(os.getenv('KODAK_TIMING_REPETITIONS', '3'))
WARMUP_REPETITIONS = int(os.getenv('KODAK_WARMUP_REPETITIONS', '1'))
COMPUTE_LPIPS = os.getenv('KODAK_COMPUTE_LPIPS', '1') == '1'
SEED = int(os.getenv('KODAK_BENCHMARK_SEED', '42'))
REPAIRED_EXPERIMENT_ROOT = os.getenv('SEMCOM_EXPERIMENT_ROOT')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print({'contributor': CONTRIBUTOR, 'run_id': RUN_ID, 'device': DEVICE})

{'contributor': 'Nandini.P', 'run_id': 'repaired-kodak24-20260922', 'device': 'cpu'}


## 2. Download and audit Kodak-24

The downloader resumes valid files, downloads through `.part` files, checks the published byte count and canonical SHA-256, validates each RGB PNG header and dimension, and writes a local manifest. This notebook then decodes every image with Pillow. The dataset directory is intentionally ignored by Git.

In [3]:
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts' / 'download_kodak24.py'), '--output', str(KODAK_DIR)],
    check=True, cwd=PROJECT_ROOT,
)
KODAK_FILES = dataset_files(KODAK_DIR)
audit = []
for path in KODAK_FILES:
    with Image.open(path) as image:
        audit.append({'image_id': path.stem, 'width': image.width, 'height': image.height, 'mode': image.mode})
audit_df = pd.DataFrame(audit)
assert len(audit_df) == 24 and set(audit_df['mode']) == {'RGB'}
display(audit_df)
display(audit_df.groupby(['width', 'height']).size().rename('images').reset_index())

,image_id,width,height,mode
0,kodim01,768,512,RGB
1,kodim02,768,512,RGB
2,kodim03,768,512,RGB
3,kodim04,512,768,RGB
4,kodim05,768,512,RGB
5,kodim06,768,512,RGB
6,kodim07,768,512,RGB
7,kodim08,768,512,RGB
8,kodim09,512,768,RGB
9,kodim10,512,768,RGB


,width,height,images
0,512,768,6
1,768,512,18


## 3. Register methods

RAW and JPEG are ready. Add each learned model either through `EncDecCodec` (preferred for repository adapters) or `CallableCodec`. Give every checkpoint/rate/SNR a distinct `operating_point` and a checkpoint hash or version string.

For continuous JSCC, report **complex channel uses**, not bytes. For an actually serialized latent, provide `payload_bits_fn`. If a number is not meaningful, leave it empty instead of inventing a conversion.

In [4]:
METHODS = [RawRGBCodec()] + [JPEGCodec(quality) for quality in JPEG_QUALITIES]
repaired_provenance = None
if REPAIRED_EXPERIMENT_ROOT:
    from Evaluation.repaired_notebook_models import build_repaired_notebook_codecs
    repaired_methods, repaired_provenance = build_repaired_notebook_codecs(
        REPAIRED_EXPERIMENT_ROOT, PROJECT_ROOT, snr_db=10.0, base_seed=SEED,
    )
    METHODS.extend(repaired_methods)
    display(pd.DataFrame([repaired_provenance]))
else:
    print('Set SEMCOM_EXPERIMENT_ROOT to add the repaired DeepJSCC/NTSCC methods.')

# Preferred: a repository EncDecInterface adapter that is already built/loaded.
# channel = AWGNChannel()
# snr_db, rate = 10.0, 96
# METHODS.append(EncDecCodec(
#     name='MySwinJSCC', adapter=my_adapter, device=DEVICE,
#     version='checkpoint-sha256-prefix',
#     operating_point=f'rate={rate};snr_db={snr_db};snr=Es/N0',
#     encode_options={'rate': rate, 'snr_db': snr_db},
#     transport_fn=lambda latent: channel.transmit(latent, snr_db),
# ))

# Generic PyTorch example. encode_fn/decode_fn are the only required hooks.
# my_model = load_my_model(...).to(DEVICE).eval()
# METHODS.append(CallableCodec(
#     name='MyModel', version='checkpoint-sha256-prefix',
#     operating_point='rate=example;snr_db=10', device=DEVICE,
#     encode_fn=lambda x: my_model.encoder(x),
#     decode_fn=lambda z: my_model.decoder(z),
#     channel_uses_fn=lambda z: math.ceil(z[0].numel() / 2),
#     modules=[my_model],
#     flops=None,  # Fill from your model's trusted profiler if available.
# ))

keys = [(method.name, method.operating_point) for method in METHODS]
assert len(keys) == len(set(keys)), 'Duplicate method/operating-point key'
pd.DataFrame(keys, columns=['method', 'operating_point'])

D:\<local OneDrive>\Desktop\image_semcom\.venv\Lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
D:\<local OneDrive>\Desktop\image_semcom\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
D:\<local OneDrive>\Desktop\image_semcom\.venv\Lib\site-packages\torch\jit\_script.py:365: FutureWarning: `torch.jit.script_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
D:\<local OneDrive>\Desktop\image_semcom\.venv\Lib\site-packages\torch\functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\n

Encoder  dim=256, input_resolution=(256, 256), depth=1
Encoder  dim=256, input_resolution=(128, 128), depth=1
Encoder  dim=256, input_resolution=(64, 64), depth=2
Encoder  dim=256, input_resolution=(32, 32), depth=4
Decoder  dim=256, input_resolution=(16, 16), depth=4
Decoder  dim=256, input_resolution=(32, 32), depth=2
Decoder  dim=256, input_resolution=(64, 64), depth=1
Decoder  dim=256, input_resolution=(128, 128), depth=1


CONFIG RATE tensor([ 16,  32,  48,  64,  80,  96, 102, 118, 134, 160, 186, 192, 208, 224,
        240, 256])


,experiment_root,deepjscc_checkpoint_sha256,ntscc_checkpoint_sha256,deepjscc_codebook_sha256,ntscc_codebook_sha256,deepjscc_calibration,ntscc_calibration,snr_db,snr_definition,base_seed
0,D:\\Deskt...,00b33bac630555d9a0d72a76d0fea9103cef9a38df0d55...,7a8d1990dbd678ea5d3daa02c3aeaaf4459b347a96d6aa...,7924944c1201260256eea1b47bcd990d820f3586ef01a9...,e553e847e60e0371cf215cef5694ee82869b97b4babe23...,"CIFAR-10 training indices 0-7, resized to 128x128",CIFAR-10 training indices 0-499 only; cached K...,10.0,Es/N0_dB,42


,method,operating_point
0,RAW_RGB24,8_bits_per_channel
1,JPEG,quality=20;subsampling=0
2,JPEG,quality=30;subsampling=0
3,JPEG,quality=40;subsampling=0
4,JPEG,quality=50;subsampling=0
5,JPEG,quality=60;subsampling=0
6,JPEG,quality=70;subsampling=0
7,JPEG,quality=80;subsampling=0
8,JPEG,quality=90;subsampling=0
9,DeepJSCC,tile=128;snr_db=10;snr=Es/N0;seed_per_image=42...


### DeepJSCC example with the repository adapter

Uncomment and edit this example on the teammate machine that has the external source/checkpoint. Models that only accept tiles should wrap tiling inside their callable hooks; the benchmark will reject a final image whose size differs from the original.

In [5]:
# from EncDecPipeline.Models.DeepJSCC.adapter import DeepJSCCAdapter
# from EncDecPipeline.Models.DeepJSCC.channel_wrapper import DeepJSCCChannelWrapper
# DEEP_ROOT = Path(r'D:/path/to/Deep-JSCC-PyTorch')
# CHECKPOINT = DEEP_ROOT / 'out' / 'checkpoint.pth'
# deep = DeepJSCCAdapter(CHECKPOINT, DEEP_ROOT, c=19, device=DEVICE).build()
# awgn = DeepJSCCChannelWrapper()
# snr_db = 10.0
# METHODS.append(EncDecCodec(
#     'DeepJSCC', deep, device=DEVICE,
#     version=deep.parameter_summary()['checkpoint_sha256'],
#     operating_point=f'c=19;snr_db={snr_db};snr=Es/N0',
#     transport_fn=lambda latent: awgn.transmit(latent, snr_db),
# ))

## 4. Run every method on all 24 images

This can take time, especially LPIPS on CPU. GPU timing is synchronized. Memory columns report process RSS and CUDA peak allocation; parameter/storage columns report static model complexity. FLOPs remain empty unless the model owner supplies a trusted value.

In [6]:
all_rows = []
for method in METHODS:
    method_rows = benchmark_codec(
        method, KODAK_FILES, run_id=RUN_ID, contributor=CONTRIBUTOR,
        project_root=PROJECT_ROOT, timing_repetitions=TIMING_REPETITIONS,
        warmup_repetitions=WARMUP_REPETITIONS, compute_lpips=COMPUTE_LPIPS,
    )
    assert len(method_rows) == 24
    all_rows.extend(method_rows)

per_image_df = pd.DataFrame(all_rows)
counts = per_image_df.groupby(['method', 'operating_point'])['image_id'].nunique()
assert (counts == 24).all(), f'Incomplete method results:\n{counts[counts != 24]}'
display(per_image_df)

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


D:\<local OneDrive>\Desktop\image_semcom\.venv\Lib\site-packages\torchvision\models\_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
D:\<local OneDrive>\Desktop\image_semcom\.venv\Lib\site-packages\torchvision\models\_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: D:\<local OneDrive>\Desktop\image_semcom\.venv\Lib\site-packages\lpips\weights\v0.1\alex.pth


RAW_RGB24 8_bits_per_channel kodim01: PSNR=inf dB, SSIM=1.00000, total=1.79 ms


RAW_RGB24 8_bits_per_channel kodim02: PSNR=inf dB, SSIM=1.00000, total=2.52 ms


...

NTSCC codebook rc_pq4_k256_raw;cifar500_only;ldpc=1/2;qpsk;snr_db=10;seed_per_image=42+index kodim21: PSNR=25.434 dB, SSIM=0.80687, total=14597.46 ms


NTSCC codebook rc_pq4_k256_raw;cifar500_only;ldpc=1/2;qpsk;snr_db=10;seed_per_image=42+index kodim22: PSNR=27.577 dB, SSIM=0.75991, total=13984.44 ms


NTSCC codebook rc_pq4_k256_raw;cifar500_only;ldpc=1/2;qpsk;snr_db=10;seed_per_image=42+index kodim23: PSNR=28.944 dB, SSIM=0.89193, total=13856.68 ms


NTSCC codebook rc_pq4_k256_raw;cifar500_only;ldpc=1/2;qpsk;snr_db=10;seed_per_image=42+index kodim24: PSNR=23.207 dB, SSIM=0.77114, total=15024.99 ms


,schema_version,run_id,timestamp_utc,contributor,git_commit,method,method_version,operating_point,device,image_id,...,peak_gpu_allocated_mb,model_parameters,trainable_parameters,model_size_mb,flops,psnr_db,ssim,lpips,timing_repetitions,notes
0,1.0,repaired-kodak24-20260922,2026-09-22T10:45:14.309537+00:00,Nandini.P,71b5501bd21988c98a482301e9d9129768b33738,RAW_RGB24,RGB24,8_bits_per_channel,cpu,kodim01,...,None,0,0,0.0000,None,inf,1.000000,0.000000,1,
1,1.0,repaired-kodak24-20260922,2026-09-22T10:45:14.778548+00:00,Nandini.P,71b5501bd21988c98a482301e9d9129768b33738,RAW_RGB24,RGB24,8_bits_per_channel,cpu,kodim02,...,None,0,0,0.0000,None,inf,1.000000,0.000000,1,
2,1.0,repaired-kodak24-20260922,2026-09-22T10:45:15.150891+00:00,Nandini.P,71b5501bd21988c98a482301e9d9129768b33738,RAW_RGB24,RGB24,8_bits_per_channel,cpu,kodim03,...,None,0,0,0.0000,None,inf,1.000000,0.000000,1,
3,1.0,repaired-kodak24-20260922,2026-09-22T10:45:15.517386+00:00,Nandini.P,71b5501bd21988c98a482301e9d9129768b33738,RAW_RGB24,RGB24,8_bits_per_channel,cpu,kodim04,...,None,0,0,0.0000,None,inf,1.000000,0.000000,1,
4,1.0,repaired-kodak24-20260922,2026-09-22T10:45:15.884752+00:00,Nandini.P,71b5501bd21988c98a482301e9d9129768b33738,RAW_RGB24,RGB24,8_bits_per_channel,cpu,kodim05,...,None,0,0,0.0000,None,inf,1.000000,0.000000,1,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,1.0,repaired-kodak24-20260922,2026-09-22T10:59:00.673608+00:00,Nandini.P,71b5501bd21988c98a482301e9d9129768b33738,NTSCC codebook,7a8d1990dbd678ea5d3daa02c3aeaaf4459b347a96d6aa...,rc_pq4_k256_raw;cifar500_only;ldpc=1/2;qpsk;sn...,cpu,kodim20,...,None,31358531,0,131.6437,None,28.969515,0.852892,0.182240,1,Rate-conditioned PQ4 K=256 design from the rep...
308,1.0,repaired-kodak24-20260922,2026-09-22T10:59:15.695584+00:00,Nandini.P,71b5501bd21988c98a482301e9d9129768b33738,NTSCC codebook,7a8d1990dbd678ea5d3daa02c3aeaaf4459b347a96d6aa...,rc_pq4_k256_raw;cifar500_only;ldpc=1/2;qpsk;sn...,cpu,kodim21,...,None,31358531,0,131.6437,None,25.433521,0.806867,0.287019,1,Rate-conditioned PQ4 K=256 design from the rep...
309,1.0,repaired-kodak24-20260922,2026-09-22T10:59:30.144715+00:00,Nandini.P,71b5501bd21988c98a482301e9d9129768b33738,NTSCC codebook,7a8d1990dbd678ea5d3daa02c3aeaaf4459b347a96d6aa...,rc_pq4_k256_raw;cifar500_only;ldpc=1/2;qpsk;sn...,cpu,kodim22,...,None,31358531,0,131.6437,None,27.576715,0.759908,0.319514,1,Rate-conditioned PQ4 K=256 design from the rep...
310,1.0,repaired-kodak24-20260922,2026-09-22T10:59:44.453558+00:00,Nandini.P,71b5501bd21988c98a482301e9d9129768b33738,NTSCC codebook,7a8d1990dbd678ea5d3daa02c3aeaaf4459b347a96d6aa...,rc_pq4_k256_raw;cifar500_only;ldpc=1/2;qpsk;sn...,cpu,kodim23,...,None,31358531,0,131.6437,None,28.944324,0.891930,0.165984,1,Rate-conditioned PQ4 K=256 design from the rep...


In [7]:
summary_df = pd.DataFrame(summarize_rows(all_rows))
columns = [
    'method', 'operating_point', 'images', 'compression_ratio_mean',
    'bits_per_pixel_mean', 'cbr_mean', 'encode_time_ms_mean',
    'decode_time_ms_mean', 'cpu_process_time_ms_mean',
    'peak_cpu_rss_delta_mb_mean', 'peak_gpu_allocated_mb_mean',
    'model_parameters_mean', 'model_size_mb_mean', 'flops_mean',
    'psnr_db_mean', 'ssim_mean', 'lpips_mean',
]
display(summary_df[columns].sort_values(['method', 'operating_point']))

,method,operating_point,images,compression_ratio_mean,bits_per_pixel_mean,cbr_mean,encode_time_ms_mean,decode_time_ms_mean,cpu_process_time_ms_mean,peak_cpu_rss_delta_mb_mean,peak_gpu_allocated_mb_mean,model_parameters_mean,model_size_mb_mean,flops_mean,psnr_db_mean,ssim_mean,lpips_mean
0,DeepJSCC,tile=128;snr_db=10;snr=Es/N0;seed_per_image=42...,24,NaN,NaN,0.162547,82.529421,174.824954,834.635417,1.169108,None,167212.0,0.637863,None,16.435117,0.237411,0.957790
1,DeepJSCC codebook,K=256;tile=128;ldpc=1/2;qpsk;snr_db=10;seed_pe...,24,58.386854,0.411051,0.137153,234.986933,2659.504746,10281.250000,35.585449,None,167212.0,0.656418,None,20.366174,0.435433,0.651415
2,JPEG,quality=20;subsampling=0,24,41.024034,0.618022,NaN,4.587112,5.026638,49.479167,0.519531,None,0.0,0.000000,None,29.510833,0.832039,0.171837
3,JPEG,quality=30;subsampling=0,24,32.544840,0.784262,NaN,4.036638,5.001208,38.411458,0.138835,None,0.0,0.000000,None,30.909509,0.869178,0.115830
4,JPEG,quality=40;subsampling=0,24,27.679328,0.925119,NaN,4.048617,4.816883,35.807292,0.891602,None,0.0,0.000000,None,31.887646,0.890134,0.087657
5,JPEG,quality=50;subsampling=0,24,24.171917,1.061386,NaN,3.682542,4.667483,34.505208,0.340007,None,0.0,0.000000,None,32.683744,0.904827,0.069817
6,JPEG,quality=60;subsampling=0,24,21.174493,1.213189,NaN,4.297313,5.654283,41.666667,0.291829,None,0.0,0.000000,None,33.464933,0.917117,0.056440
7,JPEG,quality=70;subsampling=0,24,17.747891,1.447334,NaN,4.013325,5.000938,35.156250,0.907878,None,0.0,0.000000,None,34.544633,0.931377,0.041121
8,JPEG,quality=80;subsampling=0,24,13.886195,1.846283,NaN,4.004471,5.387158,42.968750,0.423014,None,0.0,0.000000,None,36.133526,0.947524,0.024683
9,JPEG,quality=90;subsampling=0,24,9.072809,2.799021,NaN,4.608546,6.354604,44.270833,0.330729,None,0.0,0.000000,None,39.157348,0.968242,0.008078


## 5. Append to the shared CSV

Review the tables first. Then set the switch to `True` and run once. The writer is schema-checked, locked, atomic, and idempotent for the same run/method/operating-point/image key. Pull/rebase main before running and again before pushing, because Git itself can still report a CSV merge conflict.

In [8]:
APPEND_RESULTS = os.getenv('KODAK_BENCHMARK_APPEND', '0') == '1'
if APPEND_RESULTS:
    added = append_rows_csv(CSV_PATH, all_rows)
    print(f'Appended {added} new rows to {CSV_PATH}')
else:
    print('Review complete? Set APPEND_RESULTS=True to persist these rows.')

Appended 312 new rows to D:\<local OneDrive>\Desktop\image_semcom\references\SemanticSchedulerEnd2End_team\results\kodak24_benchmark.csv


## Interpretation checklist

- Compare quality at matched rate/resource points, not only the highest PSNR.
- RAW must be lossless (`PSNR=∞`, `SSIM=1`, `LPIPS≈0`, compression ratio `1`).
- JPEG has payload bits/BPP/compression ratio but no CBR without an explicit modulation/coding chain.
- Analog JSCC normally has channel uses/CBR but no payload bits.
- Do not compare CPU and GPU latency as if they were the same hardware condition; use the `device`, version, commit, and contributor columns.
- A publishable stochastic-channel result should repeat multiple seeded trials. Use a separate run ID per trial or put the trial seed in the operating point.